# Trend analysis observations (Second Covariate Analysis) [Version updated: 07-07-2026]
## Probabilistic method

---

## How to Use This Notebook

**1. Follow the numbered steps in order.**  
Each section builds upon the previous one, from setup, data loading, and climatology computation, to event analysis and visualization.

**2. Look for <font color="orange"> Orange cells  </font> and code cells marked as <font color="lightgreen">##### (User selection) ##### </font>:** 
| <font color="orange"> Orange cells  </font> | <font color="orange"> Need user intervantion </font>|
| ----------- | ----------- |
| <font color="green">**Green cells** </font> | <font color="green">**Run automatically on user input provided in the orange cells and should not be adjusted in most cases** </font>|


**3. Run cells sequentially.**  
Start from the top and execute each cell (`Shift` + `Enter`).  

### <font color="green"> Import require packages </font>

In [ ]:
from datetime import datetime, timedelta
from c3s_event_attribution_tools import *
import xarray as xr
import pandas as pd
import geopandas as gpd
import os
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# import R libraries from WWA
ro.r('''
if (!requireNamespace("remotes", quietly = TRUE)) {
    install.packages("remotes", repos="https://cloud.r-project.org", quiet=TRUE)
}

# Force install the older, compatible version of gsl
if (!requireNamespace("gsl", quietly = TRUE)) {
    remotes::install_version("gsl", version = "2.1-8", repos = "https://cloud.r-project.org")
}

# Now install rwwa (Maris fork to ensure availablity)
remotes::install_github("maris-development/rwwa@236d9a6b4a201eca1f28001b5535341022f5aeaf") 
''')

rwwa = importr("rwwa")
%load_ext rpy2.ipython

### <font color="orange"> User specifications </font>

#### <font color="orange"> Authentication & File Setup </font>

Action Required: Ensure you have entered your **CDS API Key** in the code cell above.

Storage: Data and results will be saved to: {{your_save_directory}}.

Security: ⚠️ Never share this notebook publicly or commit it to GitHub with your API keys visible.

In [ ]:
# Directory you wish to store output files in. using ../ specifies the parent directory
CURRENT_DIRECTORY = os.getcwd() # do not touch, __file__ specifies the current directory of the file

################# (User selection) ###################
your_save_directory = os.path.abspath(os.path.join(CURRENT_DIRECTORY, "./data"))   # change ../data to your desired directory
your_api_key = ''
######################################################

### <font color="orange"> Choice of parameter </font>

In [ ]:
# Choice of parameter (Tmax, Tmean, Tmin, Precipitation)
################# (User selection) ###################
parameter = "" 
event_end = datetime(, , ) #YYYY,MM,DD 

######################################################
event_start = event_end - timedelta(days=14)
######################################################

# Get parameter configuration
config = Utils.get_parameter_config(parameter)
value_col = config["value_col"]
y_label = config["y_label"]
unit = config["unit"]
calculation = config["calculation"] 
if parameter in ["Tmax", "Tmean", "Tmin"]:
    variable = "Temperature"
    fit_type = "shift"
    method = "std"
elif parameter == "Precipitation":
    variable = "Precipitation"
    fit_type = "fixeddisp"
    method = "dispersion"

## 3.2 Check (visually) for inhomogeneity of annual event time series

### <font color='green'>File name of the annual time series from the event definition step</font>

In [ ]:
annual_timeseries_load = 'ts_ann_studyregion.nc'

ts_ann_studyregion = xr.open_dataset(os.path.join(your_save_directory, annual_timeseries_load)).to_dataframe().reset_index()
ts_ann_studyregion

- a. Pay special attention to years that might coincide with key events such as the introduction of satellites (1979)
- b. Jumps in the (annual) time series
- c. visually check for a trend or for jumps outside the confidence interval in the scale or dispersion parameter. The uncertainty interval in the plot contains a first order estimate of the uncertainty according to the chi2 quantity.
     - i. For temperature plot running standard deviation with a 15-year moving window and check for large jumps
     - ii. For precipitation plot running dispersion (stdev/mean) with a 15-year moving window and check for large jumps
- d. Decide on which years to use if data is not homogeneous (limit to use a later starting year) 

In [ ]:
ts_ann_studyregion_15y = Process.calculate_rolling_window(gdf=ts_ann_studyregion, value_col=value_col, datetime_col="year",
                                  window=15, method=method, min_periods=1, centering=True, ci=0.95)

ts_ann_studyregion_15y

In [ ]:
Plot.plot_timeserie(data=ts_ann_studyregion_15y, datetime_col="year", value_col=value_col,
               title=f"15 year running {method} of {parameter}", x_label="Date", y_label=method, line_style='solid', ci=True);

## 3.2.d. Decide on which years to use if data is not homogeneous

### <font color='orange'>Specify the preferred time range based on the plot</font>

In [ ]:
event_year = ts_ann_studyregion_15y.iloc[-1].year
year_range = (1950, event_year)

### <font color='green'> Slice the annual time series on the given time range </font>

In [ ]:
ts_ann_studyregion_subset = Utils.subset_gdf(gdf=ts_ann_studyregion, datetime_col="year", date_range=year_range)
ts_ann_studyregion_subset

## 3.3 Conclude about the quality of ERA5
Evaluate the quality of ERA5 (and potentially other datasets) for the specific event definition and implications for the study results and the years to use, to feed into the synthesis Step 6.11; document into the output table in Notes & tables and the scientific report Section 2.1. 
- a. For the region, check performance of the dataset, conclude on whether the data can be used with limited problems (“satisfactory”) or caution is necessary (“caution”) and conclude on a sentence in the scientific report.  
- b. For the use of years use the ERA5 performance document and using plots produced in the Jupyter notebook under Step 3.2: visually check whether scale (temperature) or dispersion (precipitation) parameter show a trend and only use the stationary scale/dispersion-fit if the stationarity assumption is not obviously invalid in the sense that the trend is much greater than variability. Otherwise, a non-stationary scale- or dispersion-parameter would be more appropriate (not part of OAO protocol). Continue with standard analysis and add a sentence on caution on the interpretation of results to the scientific report Section 3.1 - see also Step 3.5b.iv 
- c. OPTIONAL if local knowledge is available: decide on restricting years based on the number of observations over time, potentially overriding decisions from step b, see also Step 2.2h.vi 
- d. Write a sentence in the Scientific report Section 3.1 (see end of ‘conclusion based on performance ERA5’) on which years have been used and why. 

## 3.4 decide on which covariate to use
- The GMST will have been filled up to the previous month (last row of the gmst_monthly array printed to screen), and annual GMST is calculated using persistence (from April onwards, last row of the annual array)

In [ ]:
client = DataClient(cds_key=your_api_key, beacon_cache_url="https://beacon-era5.maris.nl/")
gmst = client.gmst_xr(time_range=(datetime(1950, 1, 1), datetime(event_year, 12, 31))) # bbox entire world, end datetime today
gmst_weights = Process.weighted_values(gmst, 't2m') # xarray does not allow as to apply the weight in the same way as pandas/geopandas, therefore we only return the weight and apply it later in the calculation

gmst_monthly = (
    gmst['t2m']
    .groupby('valid_time')
    .map(lambda da: da.weighted(gmst_weights).mean(dim=('latitude', 'longitude')))
    .drop_vars(['number', 'expver'], errors='ignore')
    .to_dataframe(name='gmst')
    .reset_index()
)
gmst_monthly

### <font color='green'> Low-pass filter GMST (4-year smoothing) </font>

The GMST covariate is smoothed with a 4-year low-pass filter For each calendar month separately, we take a 4-year mean and label it on the **3rd year** of its window, i.e. `value[year] = mean(year-2, year-1, year, year+1)`. The 12 smoothed months are then averaged into a single annual GMST value.

At the start and end of the record the window is naturally shorter (e.g. the
first year uses only the first two available years, the event year uses the
years up to the latest data), so **no years are lost**.

In [ ]:

gmst_monthly_filled = gmst_monthly.copy()

full_range = pd.date_range(
    start=gmst_monthly_filled['valid_time'].min(),
    end=pd.Timestamp(year=int(event_year) + 1, month=12, day=1),   # one extra year ahead
    freq="MS",
)
gmst_monthly_filled = (
    gmst_monthly_filled.set_index('valid_time')
    .reindex(full_range)                 # missing/future months become NaN
    .rename_axis('valid_time').reset_index()
)
gmst_monthly_filled = gmst_monthly_filled.sort_values('valid_time')

# trailing 4-year mean per calendar month -> value at year Z = mean{Z-3..Z}
gmst_monthly_filled['gmst_trailing'] = (
    gmst_monthly_filled
    .groupby(gmst_monthly_filled['valid_time'].dt.month)['gmst']
    .transform(lambda s: s.rolling(window=4, min_periods=1).mean())
)

# relabel to 3rd-year convention: value[y] = trailing[y+1] = window{y-2..y+1}
gmst_monthly_filled['gmst_lowpass'] = (
    gmst_monthly_filled
    .groupby(gmst_monthly_filled['valid_time'].dt.month)['gmst_trailing']
    .shift(-1)
)
#delete rows with valid_time.year > event_year
gmst_monthly_filled = gmst_monthly_filled[gmst_monthly_filled['valid_time'].dt.year <= int(event_year)]
gmst_monthly_filled

In [ ]:
# Drop the extra padded year (event_year+1) that only existed to supply y+1.
gmst_lowpass = gmst_monthly_filled[['valid_time', 'gmst_lowpass']].copy()
gmst_lowpass['year'] = gmst_lowpass['valid_time'].dt.year
gmst_lowpass = gmst_lowpass[gmst_lowpass['year'] <= int(event_year)]
gmst_lowpass

### <font color='green'> Annual GMST covariate </font>
Average the 12 smoothed monthly values within each calendar year.

In [ ]:
# Average the 12 smoothed months into the annual covariate.
gmst_yearly = (
    gmst_lowpass
    .groupby('year')['gmst_lowpass']
    .mean()
    .reset_index()
    .rename(columns={'gmst_lowpass': 'gmst'})
    .dropna(subset=['gmst'])
)
gmst_yearly

### <font color='green'> Merging yearly GMST with annual time series </font>

In [ ]:
merged_gmst = pd.merge(ts_ann_studyregion_subset, gmst_yearly, left_on="year", right_on="year", how="inner") 
ref_val = merged_gmst.iloc[-1]['gmst'] # calculate anomaly relative to event year
merged_gmst_anomaly = merged_gmst.copy()
merged_gmst_anomaly["gmst"] = merged_gmst_anomaly['gmst'] - ref_val
merged_gmst_anomaly

### <font color='green'> ENSO index calculation (Download SST for the ENSO index) </font>

Downloads ERA5 monthly sea-surface temperature over the tropical band (20°S–20°N, all longitudes), the area needed to build the Niño 3.4 index.


In [ ]:
client = DataClient(cds_key=your_api_key, beacon_cache_url="https://beacon-era5.maris.nl/")
sst = client.fetch_era5_monthly_single_levels_xr(variable=Variable.ERA5MonthlySingleLevel.sea_surface_temperature, bbox = (-180, -20, 180, 20), time_range=(datetime(1950, 1, 1), event_end)) #bbox (min_lon, min_lat, max_lon, max_lat)
sst = sst.sortby('latitude') 
sst = sst.sortby('longitude') 

### <font color='green'> Compute the detrended Niño 3.4 index </font>
Builds the monthly ENSO covariate:
1. Area-weighted SST average over the **Niño 3.4** box (5°S–5°N, 170°W–120°W) and over the **wider tropics** (20°S–20°N).
2. Anomalies of each relative to the **1991–2020** climatology.
3. Niño 3.4 anomaly **minus** the tropical-mean anomaly, removing the long-term warming signal so only the ENSO variability remains.

In [ ]:
# Subset the data to the Nino 3.4 region and the tropical region
sst_nino = sst.sel(latitude=slice(-5, 5), longitude=slice(-170, -120))
sst_tropics = sst.sel(latitude=slice(-20, 20), longitude=slice(-180, 180))

# Spatial mean of each region, climatologies and anomalies
weights = Process.weighted_values(sst_nino, value_col=None, lat_col='latitude')
nino_ts = sst_nino.weighted(weights).mean(['latitude', 'longitude']).sortby("valid_time")
nino_clim = nino_ts.sel(valid_time=slice("1991-01-01", "2020-12-31")).groupby("valid_time.month").mean()
nino_anom = nino_ts.groupby("valid_time.month") - nino_clim


weights_tropics = Process.weighted_values(sst_tropics, value_col=None, lat_col='latitude')
tropics_ts = sst_tropics.weighted(weights_tropics).mean(['latitude', 'longitude']).sortby("valid_time")
tropics_clim = tropics_ts.sel(valid_time=slice("1991-01-01", "2020-12-31")).groupby("valid_time.month").mean()
tropics_anom = tropics_ts.groupby("valid_time.month") - tropics_clim

# Detrend — subtract the broad tropical mean to remove long-term warming signal
enso_monthly = nino_anom - tropics_anom
enso_monthly = enso_monthly["sst"].sortby("valid_time")
enso_monthly_df = enso_monthly.to_dataframe().reset_index()
enso_monthly_df = enso_monthly_df[['valid_time', 'sst']].rename(columns={'valid_time': 'date', 'sst': 'nino'})
enso_monthly_df

### <font color='green'> Plot the Niño 3.4 index (raw vs detrended) </font>

In [ ]:
nino_raw_df = nino_anom["sst"].to_dataframe().reset_index()[["valid_time", "sst"]] \
                 .rename(columns={"sst": "nino_raw"})
nino_plot = nino_raw_df.merge(enso_monthly_df.rename(columns={"date": "valid_time",
                                                             "nino": "nino_detrended"}),
                              on="valid_time", how="inner")

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(nino_plot["valid_time"], nino_plot["nino_raw"], lw=1, label="Niño3.4 anomaly (raw)")
ax.plot(nino_plot["valid_time"], nino_plot["nino_detrended"], lw=1.2, label="Niño3.4 (detrended)")
ax.axhline(0, color="grey", lw=0.8)
ax.axvline(pd.Timestamp(event_end), color="magenta", ls="--", lw=0.5, label="event")
ax.set_xlabel("Year"); ax.set_ylabel("Niño3.4 index")
ax.set_title("Niño 3.4: raw vs detrended"); ax.legend(); ax.grid(True)

### <font color='orange'> Select the ENSO season and year (User input) </font>
This is the **only ENSO cell you need to adjust**. It picks which months represent ENSO for your event:
- **`target_month`** — central month of the ENSO season (1=Jan … 12=Dec).
- **`rolling_window`** — number of months averaged (1 = single month, 3 = 3-month season such as OND).
- **`center`** — `True` centres the window on `target_month` (e.g. OND for Nov); `False` uses a trailing window (e.g. SON for Nov).
- **`nino_year_shift`** — shifts the Niño year by ±n so the most recent ENSO season can be matched to the event year.

Check the printed *"most recent data available"* message and the table below to confirm the year stamp is the one you expect before continuing. valid_time can be used as validation of the selected target month

In [ ]:
######################### User selection for ENSO analysis #########################
# ADJUST THESE TWO VARIABLES FOR YOUR ANALYSIS:
target_month =  1   # 1=Jan, 2=Feb, ..., 11=Nov, 12=Dec (e.g., 11 for November)
rolling_window = 3   # 1 for a single month, 2 for 2-month average, 3 for a 3-month average (e.g., OND centered on Nov), the rolling is applying center=True. If you prefer a trailing window (e.g., SON for Nov), change center to False in the rolling mean calculation below.
center = True  # Set to True for centered rolling mean, False for trailing rolling mean
nino_year_shift = 0 # Shift the year for the Nino index if needed
####################################################################################

latest_date = pd.to_datetime(enso_monthly.valid_time.max().values)
print(f"Notice: The most recent data available is {latest_date.strftime('%B %Y')}.\n")


# Apply rolling mean based on the window choice
if rolling_window > 1:
    # Note: center=True means a 3-month window for Month 11 calculates Oct-Nov-Dec. 
    # If you prefer a trailing window (e.g., SON for Nov), change center to False.
    enso_rolling = enso_monthly.rolling(valid_time=rolling_window, center=center, min_periods=1).mean()
else:
    enso_rolling = enso_monthly

enso_filtered = enso_rolling.sel(valid_time=enso_rolling.valid_time.dt.month == target_month)
enso_df = enso_filtered.to_dataframe().reset_index()
enso_df['year'] = enso_df['valid_time'].dt.year
enso_df = enso_df.rename(columns={'sst': 'nino'}).dropna(subset=['nino'])
enso_df = enso_df[['year', 'valid_time', 'nino']]
enso_df['year'] = enso_df['year'] + nino_year_shift
enso_df

### <font color='green'> Add the Niño covariate to the time series </font>
Merges the selected yearly Niño index into the annual table on `year`. Years with no Niño value get `NaN`.

In [ ]:
enso_df = enso_df[['year', 'nino']]
merged_gmst_anomaly = pd.merge(merged_gmst_anomaly, enso_df, on="year", how="left")

# A Nino year shift leaves the boundary year(s) without a matching Nino value (NaN).
if nino_year_shift != 0:
    merged_gmst_anomaly = merged_gmst_anomaly.dropna(subset=['nino']).reset_index(drop=True)

# Save the final merged DataFrame with GMST anomaly and Nino index to a CSV file
merged_gmst_anomaly.to_csv("./data/cov_observations.csv", index=False)
merged_gmst_anomaly

### <font color='green'> Correlation </font>


In [ ]:
%%R -i value_col -i merged_gmst_anomaly 

df <- merged_gmst_anomaly

nsamp <- 1000
nino_corr <- data.frame(t(c(cor(df$nino, df[[value_col]]), quantile(sapply(1:nsamp, function(i) cor(df[sample(1:nrow(df), replace = T), c("nino", value_col)])[1,2]), c(0.025, 0.975), na.rm = T))))
colnames(nino_corr) <- c("nino_corr_est", "nino_corr_lower", "nino_corr_upper")
cat(sprintf("Nino3.4 vs %s: %.3f (%.3f to %.3f)\n", value_col, nino_corr$nino_corr_est, nino_corr$nino_corr_lower, nino_corr$nino_corr_upper))

## 3.5 Apply statistical method
information for decisions on fit properties:
- a. fit data to statistical model, and check, at least visually, that this fit agrees with the observed data points. Show this in a figure. The event magnitude is shown in figures as a magenta point or a magenta dashed horizontal line. Important: when studying cold extremes, in the user input box under “Please specify the following variables”, set lower=TRUE ! This ensures that the lowest values of the distribution are analysed instead of the highest values.
    -  i. Gauss - appropriate for moderate extremes with low return periods that are not in the tail, also known as the Normal distribution. Threshold μ (mu), scale parameter σ (sigma). Usually used for seasonal averages. Set dist = “norm” to choose this distribution.
    -  ii. GEV - appropriate for all extremes including those with high return periods. The extreme for each entry in the timeseries must however be selected from a longer temporal block of fixed size.  Location parameter μ, scale parameter σ, shape parameter ξ. Usually used for 1-day to 14-day maxima or minima of a specified longer block period of a specific month or season or the full year (Jan-Dec or Jul-Jun). Set dist = “gev” to choose this distribution. 
    - iii. If the fit (Gauss or GEV) does not agree well with the observed data points, the other distribution may be tested and used instead. 
        - 1. The AIC is a measure of how well the model (distribution) fits the data. A lower AIC is better. AIC values can only be compared when using exactly the same data but for different statistical models (e.g. Gauss or GEV, or the fit using one or two covariates). 
        - 2. If both fits do not agree well with the observed data points, consider changing to an in-depth study to test a Lognormal distribution; possibly Lognormal is better for precipitation (not coded). 
- b. shift/scale with GMST 
    - i. for temperature extremes the distribution shifts due to global warming without changing the shape (μ changes, proportional to the smoothed GMST; σ and ξ are constant).  
      - Note that for cold events we look at the lower tail of the distribution 
    - ii. for precipitation extremes the distribution scales without changing the shape (dispersion parameter (σ/μ) scales, proportional to the smoothed GMST; ξ is constant). 
    - iii. In-depth only (not coded) for precipitation with Lognormal use shift 
    - iv. Use the visual check of scale or dispersion parameter (done in workflow Step 3.3b): use the stationary scale/dispersion-fit if the stationarity assumption is not obviously invalid. Otherwise, using a non-stationary scale- or dispersion-parameter would be more appropriate but it is not currently available (not coded), and therefore a sentence on caution on the results should be added to the scientific report Section. 3.1 

## <font color='green'>Check inputs for the plots<font>

In [ ]:
event_year = merged_gmst.iloc[-1]['year']
bounds = np.array([merged_gmst_anomaly[value_col].min(), merged_gmst_anomaly[value_col].max()])
event_value = merged_gmst_anomaly[value_col].iloc[-1]
print('Event year:', event_year)
print('Event value:', event_value)
print('Plot y-limits:', bounds)

## <font color='orange'>Please specify the following variables<font>

In [ ]:
%%R -i event_year -i value_col -i fit_type
################# (User selection) ###################
dist = "gev" #choose between "norm" or "gev" 
covnm = c("gmst", "nino") #choose between "gmst", "nino" or both
lower = FALSE #choose TRUE for cold events
cooling_offset = 1.3 #fixed to 1.3 for now, can be changed at later stage if required
######################################################

## <font color='green'>Plotting of trends<font>

In [ ]:
%%R -i merged_gmst_anomaly 

df <- merged_gmst_anomaly

mdl_gmst <- fit_ns(dist = dist, type = fit_type, data = df, varnm = value_col, covnm = covnm[1], lower = lower, ev_year = event_year)
mdl_gmstnino <- fit_ns(dist = dist, type = fit_type, data = df, varnm = value_col, covnm = covnm, lower = lower, ev_year = event_year)

# the factual climate should have the GMST of the year in which the event occurred
cov_factual <- df[df$year == event_year, c("gmst","nino"), drop = F]

# the counterfactual climate can represent any alternative climate 
cov_counterfactual <- data.frame(rbind("pi" = c("gmst" = cov_factual$gmst - cooling_offset, "nino" = cov_factual$nino),
                           "neut" = c("gmst" = cov_factual$gmst, "nino" = 0),
                           "pineut" = c("gmst" = cov_factual$gmst - cooling_offset, "nino" = 0))) 

In [ ]:
ylab = (f"{parameter} ({unit})")

In [ ]:
%%R -i ylab -w 1920 -h 960 -r 200 -i bounds
# what does the fitted trend look like over time?
# `add_loess = T` will add a nonparametric smoother - use this to check whether the fitted model captures the observed trend 
prep_window(c(1,2))
pad  <- 0.05 * diff(bounds)
ylim <- c(bounds[1] - pad, bounds[2] + pad)
plot_trend(mdl_gmst, add_loess = T, ylim = ylim, main = "gmst")
plot_trend(mdl_gmstnino, add_loess = T, ylim = ylim, main = "gmst + nino")

png("./data/fig_obs-trend_era5.png", height = 960, width = 1920, res = 200)
  par(mfrow = c(1,2))
  plot_trend(mdl_gmst,    add_loess = T, ylim = ylim, ylab = ylab, lwd = 2, main = "gmst")
  plot_trend(mdl_gmstnino, add_loess = T, ylim = ylim, ylab = ylab, lwd = 2, main = "gmst + nino")
invisible(dev.off())


In [ ]:
%%R -i ylab -w 2880 -h 1400 -r 200 -i bounds
# what does the fitted trend look like vs GMST?
prep_window(c(1,3))
pad  <- 0.05 * diff(bounds)
ylim <- c(bounds[1] - pad, bounds[2] + pad)
plot_covtrend(mdl_gmst, xcov = covnm[1], add_loess = T, ylim = ylim, main = "GMST-only model")
plot_covtrend(mdl_gmstnino, xcov = covnm[1], add_loess = T, ylim = ylim, main = "Two-covariate model: GMST trend")
plot_covtrend(mdl_gmstnino, xcov = covnm[2], add_loess = T, ylim = ylim, main = "Two-covariate model: Nino trend")

png("./data/fig_obs-gmsttrend_era5.png", height = 1400, width = 2880, res = 200)
  par(mfrow = c(1,3))
  plot_covtrend(mdl_gmst,    xcov = covnm[1], add_loess = T, ylim = ylim, ylab = ylab, lwd = 2, main = "GMST-only model")
  plot_covtrend(mdl_gmstnino, xcov = covnm[1], add_loess = T, ylim = ylim, ylab = ylab, lwd = 2, main = "Two-covariate model: GMST trend")
  plot_covtrend(mdl_gmstnino, xcov = covnm[2], add_loess = T, ylim = ylim, ylab = ylab, lwd = 2, main = "Two-covariate model: Nino trend")
invisible(dev.off())


In [ ]:
%%R -i ylab -w 850 -h 1500 -r 200 -i bounds
# How well does the model fit the data?
# the points should be close to the line - if they're not within the shaded region, the model is a poor fit
rp_xmax <- 500   # max return period (years) shown on x-axis
ylim <- c(bounds[1] - 0.05*diff(bounds), bounds[2] + 0.25*diff(bounds)) 
prep_window(c(3,1))
plot_returnlevels(mdl_gmstnino, cov_f = cov_factual, cov_cf = cov_counterfactual["pi",,drop=F],     xlim = c(1,rp_xmax), ylim = ylim, main = "PI climate (same ENSO)")
plot_returnlevels(mdl_gmstnino, cov_f = cov_factual, cov_cf = cov_counterfactual["neut",,drop=F],   xlim = c(1,rp_xmax), ylim = ylim, main = "Present, ENSO-neutral")
plot_returnlevels(mdl_gmstnino, cov_f = cov_factual, cov_cf = cov_counterfactual["pineut",,drop=F], xlim = c(1,rp_xmax), ylim = ylim, main = "PI + ENSO-neutral")

png("./data/fig_obs-returnlevels_era5_nino.png", height = 1500, width = 850, res = 200)
  par(mfrow = c(3,1))
  plot_returnlevels(mdl_gmstnino, cov_f = cov_factual, cov_cf = cov_counterfactual["pi",,drop=F],     xlim = c(1,rp_xmax), ylim = ylim, ylab = ylab, main = "PI climate (same ENSO)")
  plot_returnlevels(mdl_gmstnino, cov_f = cov_factual, cov_cf = cov_counterfactual["neut",,drop=F],   xlim = c(1,rp_xmax), ylim = ylim, ylab = ylab, main = "Present, ENSO-neutral")
  plot_returnlevels(mdl_gmstnino, cov_f = cov_factual, cov_cf = cov_counterfactual["pineut",,drop=F], xlim = c(1,rp_xmax), ylim = ylim, ylab = ylab, main = "PI + ENSO-neutral")
invisible(dev.off())


In [ ]:
%%R -w 1200 -h 500 -r 100 -i bounds
prep_window(c(1,2))
ylim <- c(bounds[1] - 0.05*diff(bounds), bounds[2] + 0.25*diff(bounds))
rp_xmax <- 1000

plot_returnlevels(mdl_gmst,    cov_f = cov_factual, cov_cf = cov_counterfactual["pi",,drop=F],    xlim = c(1,rp_xmax), ylim = ylim, main = "GMST only")
plot_returnlevels(mdl_gmstnino, cov_f = cov_factual, cov_cf = cov_counterfactual["pineut",,drop=F], xlim = c(1,rp_xmax), ylim = ylim, main = "GMST + Nino")

png("./data/fig_obs-returnlevels_era5.png", height = 500, width = 1200, res = 100)
par(mfrow = c(1,2))
plot_returnlevels(mdl_gmst,     cov_f = cov_factual, cov_cf = cov_counterfactual["pi",,drop=F],     xlim = c(1,rp_xmax), ylim = ylim, main = "GMST only",    ylab = ylab)
plot_returnlevels(mdl_gmstnino, cov_f = cov_factual, cov_cf = cov_counterfactual["pineut",,drop=F], xlim = c(1,rp_xmax), ylim = ylim, main = "GMST + Nino",  ylab = ylab)
invisible(dev.off())

In [ ]:
%%R -i ylab -w 1920 -h 960 -r 200 -i bounds
# what does the fitted trend look like vs GMST?
prep_window(c(1,2))
pad  <- 0.05 * diff(bounds)
ylim <- c(bounds[1] - pad, bounds[2] + pad)
plot_covtrend(mdl_gmstnino, xcov = covnm[1], add_loess = T, ylim = ylim, main = "GMST trend")
plot_covtrend(mdl_gmstnino, xcov = covnm[2], add_loess = T, ylim = ylim, main = "Nino trend")

png("./data/fig_obs-trend_era5_2cov.png", height = 960, width = 1920, res = 200)
  par(mfrow = c(1,2))
  plot_covtrend(mdl_gmstnino, xcov = covnm[1], add_loess = T, ylim = ylim, ylab = ylab, lwd = 2, main = "GMST trend")
  plot_covtrend(mdl_gmstnino, xcov = covnm[2], add_loess = T, ylim = ylim, ylab = ylab, lwd = 2, main = "Nino trend")
invisible(dev.off())


In [ ]:
%%R -w 1200 -h 500 -r 100 -i bounds
prep_window(c(1,2))
ylim <- c(bounds[1] - 0.05*diff(bounds), bounds[2] + 0.25*diff(bounds))
rp_xmax <- 1000

plot_returnlevels(mdl_gmstnino,    cov_f = cov_factual, cov_cf = cov_counterfactual["pi",,drop=F],    xlim = c(1,rp_xmax), ylim = ylim, main = "PI climate (same ENSO)")
plot_returnlevels(mdl_gmstnino, cov_f = cov_factual, cov_cf = cov_counterfactual["neut",,drop=F], xlim = c(1,rp_xmax), ylim = ylim, main = "Present, ENSO-neutral")

png("./data/fig_obs-returnlevels_era5_2cov.png", height = 500, width = 1200, res = 100)
par(mfrow = c(1,2))
plot_returnlevels(mdl_gmstnino,     cov_f = cov_factual, cov_cf = cov_counterfactual["pi",,drop=F],     xlim = c(1,rp_xmax), ylim = ylim, main = "PI climate (same ENSO)",    ylab = ylab)
plot_returnlevels(mdl_gmstnino, cov_f = cov_factual, cov_cf = cov_counterfactual["neut",,drop=F], xlim = c(1,rp_xmax), ylim = ylim, main = "Present, ENSO-neutral",  ylab = ylab)
invisible(dev.off())


## <font color='green'>Saving figure </font>

### Including two plots: 1) the trend over time, 2) GEV fit
#### Includes logos for uptake in report

### <font color="orange"> You can provide titles for both the fitted trend over time and the GEV fit and the figure as a whole </font>

In [ ]:
#################### (User selection) ##################
# Paths to your saved PNGs
file1 = "./data/fig_obs-trend_era5_2cov.png"
file2 = "./data/fig_obs-returnlevels_era5_2cov.png"
titlefig1 = None
titlefig2 = None
title = None
########################################################

fig, axs, logo_ax = Plot.plot_two_figures(file_left=file1, file_right=file2, add_logos=True, title_fig1=titlefig1, title_fig2=titlefig2, title=title, orientation='vertical')
fig.savefig(os.path.join(your_save_directory, "trend_analysis_combined_2cov.png"), dpi=300, bbox_inches='tight')
print(f"Figure saved to {os.path.join(your_save_directory, 'trend_analysis_combined_2cov.png')}")


In [ ]:
#################### (User selection) ##################
# Paths to your saved PNGs
file1 = "./data/fig_obs-trend_era5.png"
file2 = "./data/fig_obs-returnlevels_era5.png"
titlefig1 = None
titlefig2 = None
title = None
########################################################

fig, axs, logo_ax = Plot.plot_two_figures(file_left=file1, file_right=file2, add_logos=True, title_fig1=titlefig1, title_fig2=titlefig2, title=title, orientation='vertical')
fig.savefig(os.path.join(your_save_directory, "trend_analysis_combined.png"), dpi=300, bbox_inches='tight')
print(f"Figure saved to {os.path.join(your_save_directory, 'trend_analysis_combined.png')}") 

## 3.6 Observed probability and trend detection:
Results will be written to res-obs_era5.cvs. This file will be used by the Jupyter Notebooks for model validation and analysis. 
- a. Note down decision on the statistical method (see Step 3.5) in “Table event definition” in the Notes & Tables document 
- b. Note dGMST (see variable ‘cooling_offset’ in the Jupyter notebook) in “Table observational magnitude and return period” in Notes & Tables, i.e. the difference between GMSTevent and GMSTpast, based on the preindustrial climate of 1850-1900, in 1 decimal, e.g. 1.3 (this is the value in 2025). C3S defines the 1850–1900 pre‑industrial global mean surface temperature to be 0.88 °C below the 1991–2020 ERA5 global average. C3S will inform us when the dGMST value should be updated - check once a year that this value is still valid (listed in tasklist as an annual task). 
- c. Note down in “Table observational datasets considered” in Notes & Tables all observational/reanalysis datasets that passed the quality checks and for which years (for years see Step 3.2c and 3.3). Likely only ERA5 has been considered, or potentially also a station data time series. 
- d. Note down/save variability (σ) or dispersion (σ/μ) and shape parameter ξ (for GEV) of the fit (the Jupyter Notebook saves these to res-obs_era5.csv which is sufficient). 
- e. Note down return period in the current climate (printed in Jupyter Notebook and in res-obs_era5.csv columns return_period_est, return_period_lower and return_period_upper)), i.e. in Yevent, e.g., 2025, and decide on the single rounded value of the return period that will be used for communication purposes and the model analyses – write this in “Table observational magnitude and return period” in Notes & Tables. Rounding should correspond to the order of magnitude, e.g., 13.876 (minimum 8.124 to maximum 20.573) could be rounded to 14 (8-21). 
    - i. Rounding general guideline:
        - 1. Round to 1 or 2 significant figures
        - 2. Between 10 and 50 round to nearest 5
    - ii. if necessary, make decision on using e.g. lower bound in case of too extreme a return period. Note down in Notes & tables “Table event definition” whether to use the automated return period from res-obs_era5.csv or the lower bound (then indicate the lower bound). Then use this value in the model analysis as well and write a sentence to the scientific report to explain that the best estimate of the return period is too extreme to be useful (>1000 years) and therefore a more meaningful standard return period of e.g. 1000 years has been used for the analysis. 
- f. Note down the threshold value “Event magnitude” corresponding to the event in "Table observational magnitude and return period" in Notes & Tables (has been written to res-obs_era5.csv column event_magnitude_est). 
- g. Note down/save probability ratio, PR, for dGMST, and change in intensity, ΔI, (absolute value for temperature, relative % change for precipitation, i.e., (present-past)/past x 100%. The Jupyter Notebook saves this to res-obs_era5.csv columns PR_est, PR_lower, PR_upper and dI_abs_est, dI_abs_lower, dI_abs_upper for absolute change or dI_rel_est, dI_rel_lower, dI_rel_upper for relative change, which is sufficient.) 
- h. In the case of a PR <1, which will most often occur for cold extremes which are becoming less likely with climate change, the inverse PR (as well as inverse uncertainty range) should also be noted – the inverse PR will be calculated in Step 6.6a.i. Statements based on the inverse PR (values larger than 1) are much easier to communicate, see Step 6.9a.i. To avoid manual conversion, conversion can be done later in the Jupyter Notebook in the Synthesis step. 

In [ ]:
%%R
# use the built-in function to bootstrap the model results

boot_res <- boot_ci(mdl_gmstnino, cov_f = cov_factual, cov_cf = cov_counterfactual)

colnames(boot_res)[colnames(boot_res) == "2.5%"]  <- "lower"
colnames(boot_res)[colnames(boot_res) == "97.5%"] <- "upper"

# transpose to get the correct output format
boot_res_t <- data.frame(
    t(
        unlist(
            lapply(rownames(boot_res), function(rn) {
                setNames(
                    as.numeric(boot_res[rn, ]),
                    paste0(rn, "_", colnames(boot_res))
                )
            })
        )
    ),
    row.names = "era5"
)
final_res <- cbind(boot_res_t, nino_corr)

# save as a .csv to look at them later
# this .csv is used for the Synthesis notebook
write.csv(final_res, "./data/res-obs_era5.csv")

In [ ]:
%%R -i event_value
rp_factual  <- return_period(mdl_gmstnino, x = event_value, fixed_cov = cov_factual)
rp_pi       <- return_period(mdl_gmstnino, x = event_value, fixed_cov = cov_counterfactual["pi",,drop=F])
rp_neut     <- return_period(mdl_gmstnino, x = event_value, fixed_cov = cov_counterfactual["neut",,drop=F])
rp_pineut   <- return_period(mdl_gmstnino, x = event_value, fixed_cov = cov_counterfactual["pineut",,drop=F])

cat(sprintf("Return period (factual):               %s (%f - %f)\n", (rp_factual), boot_res["return_period", "lower"], boot_res["return_period", "upper"]))
cat(sprintf("Return period (PI, same ENSO):         %f\n", rp_pi))
cat(sprintf("Return period (present, ENSO-neutral): %f\n", rp_neut))
cat(sprintf("Return period (PI + ENSO-neutral):     %f\n", rp_pineut))

In [ ]:
%%R
cat(sprintf("number of rows (GMST):       %d\n", nrow(na.omit(df[, c(value_col, "gmst")]))))
cat(sprintf("number of rows (GMST+Nino):  %d\n", nrow(na.omit(df[, c(value_col, "gmst", "nino")]))))

mdl_gmst_gev      <- fit_ns("gev",  fit_type, df, value_col, covnm[1], lower = lower)
mdl_gmst_norm     <- fit_ns("norm", fit_type, df, value_col, covnm[1], lower = lower)
mdl_gmstnino_gev  <- fit_ns("gev",  fit_type, df, value_col, covnm,    lower = lower)
mdl_gmstnino_norm <- fit_ns("norm", fit_type, df, value_col, covnm,    lower = lower)

cat(sprintf("AIC (GMST,      GEV):   %f\n", aic(mdl_gmst_gev)))
cat(sprintf("AIC (GMST,      Norm):  %f\n", aic(mdl_gmst_norm)))
cat(sprintf("AIC (GMST+Nino, GEV):   %f\n", aic(mdl_gmstnino_gev)))
cat(sprintf("AIC (GMST+Nino, Norm):  %f\n", aic(mdl_gmstnino_norm)))